In [6]:
import time

import bs4
import pandas as pd
import requests
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import gnews
import yfinance as yf
from gnews import GNews
from lxml.html.diff import href_token
import bs4 as bs
import json
import csv
import nltk

In [2]:
news=GNews(start_date=(2019,11,4), end_date=(2024,11,2),max_results=1000)
ns=news.get_news("Microsoft")

C:\Users\Asus\anaconda3\Lib\site-packages\gnews\gnews.py:218: UserWarning: Only searches using get_news support date ranges. Start and end dates will be ignored.
  return self._get_news_more_than_100(key)


In [1]:
news_df=pd.DataFrame(data=ns)
news_df.head()

NameError: name 'pd' is not defined

In [4]:
news_df.shape

(1000, 5)

In [40]:
temp=news_df
temp["full_text"] = (temp["title"].fillna("") + "" +temp["description"].fillna(""))

In [41]:
temp.head()

,title,description,published date,url,publisher,full_text
0,Microsoft Bans Term ‘Microslop’ From Official ...,Microsoft Bans Term ‘Microslop’ From Official ...,"Mon, 02 Mar 2026 19:40:56 GMT",https://news.google.com/rss/articles/CBMikwFBV...,"{'href': 'https://gizmodo.com', 'title': 'Gizm...",Microsoft Bans Term ‘Microslop’ From Official ...
1,"Microsoft says stop calling it Microslop, or y...","Microsoft says stop calling it Microslop, or y...","Mon, 02 Mar 2026 15:35:00 GMT",https://news.google.com/rss/articles/CBMiowFBV...,"{'href': 'https://www.pcworld.com', 'title': '...","Microsoft says stop calling it Microslop, or y..."
2,Why the Microsoft 365 Copilot bug matters for ...,Why the Microsoft 365 Copilot bug matters for ...,"Mon, 02 Mar 2026 15:37:32 GMT",https://news.google.com/rss/articles/CBMihwFBV...,"{'href': 'https://www.foxnews.com', 'title': '...",Why the Microsoft 365 Copilot bug matters for ...
3,Microsoft ends data center non-disclosure agre...,Microsoft ends data center non-disclosure agre...,"Tue, 03 Mar 2026 19:22:00 GMT",https://news.google.com/rss/articles/CBMi6wFBV...,"{'href': 'https://www.detroitnews.com', 'title...",Microsoft ends data center non-disclosure agre...
4,Microsoft’s big developer conference returns t...,Microsoft’s big developer conference returns t...,"Tue, 03 Mar 2026 17:00:00 GMT",https://news.google.com/rss/articles/CBMiggFBV...,"{'href': 'https://www.theverge.com', 'title': ...",Microsoft’s big developer conference returns t...


In [48]:
temp.iloc[2,1]

'Why the Microsoft 365 Copilot bug matters for data security  Fox News'

In [8]:
news_df.to_csv(r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\MSFT_news.csv")

In [4]:
subreddit=["https://www.reddit.com/r/wallstreetbets/","https://www.reddit.com/r/stocks/"]

In [3]:
'''def scrape_reddit(subreddit:str,total_posts:int):

    all_data=[]         #main list
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}     #framework

    for index,url in enumerate(subreddit):
        sr_name=url.split("/")[-2]      #names of subreddit
        after=None
        posts=[]
        print(sr_name)

        while len(all_data) < total_posts:

            params = {"limit": 100}
            if after:
                params["after"] = after

            try:
                response=requests.get(f"https://www.reddit.com/r/{sr_name}/.json",headers=headers,timeout=15,params=params)
                res_json=response.json()
                print(res_json)
                data = res_json["data"]["children"]
                response.raise_for_status()


                for post in data:
                    p = post["data"]
                    posts.append({"subreddit": sr_name,"title": p["title"],"url": "https://reddit.com" + p["permalink"],"score": p["score"],"num_comments": p["num_comments"],"created": p["created_utc"]})

                after = res_json["data"]["after"]

                if after is None:
                    break
                time.sleep(1)

                all_data.append(posts)


            except Exception as e:
                print(e)

    return all_data

def save_scraped_data(data, filename_json, filename_csv):
    if not data:
        print(f'No data')
        return


    try:
        with open(filename_json, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=True)
        print(f'Topics saved: {filename_json}')
    except Exception as e:
        print("ERROR:",e)



    try:
        with open(filename_csv, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Subreddit', 'Type', 'Title', 'URL'])

            for subreddit_data in data:
                subreddit = subreddit_data['name']

                for topic in subreddit_data.get('topics', []):
                    writer.writerow([subreddit, topic['type'], topic['title'], ''])

                for discussion in subreddit_data.get('discussions', []):
                    writer.writerow([subreddit, discussion['type'], discussion['title'], discussion['url']])

            print(f'All topics saved to files!')
    except Exception as e:
        print("Error:",e)

a=scrape_reddit(subreddit,5)
save_scraped_data(a,"smth.json","smth.csv")
a'''

In [22]:
'''def scrape_reddit(subreddit):
    all_data=[]
    for index,url in enumerate(subreddit):
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}
        sr_name=url.split("/")[-2]
        print(sr_name)

        try:
            response=requests.get(url,headers=headers,timeout=15)
            response.raise_for_status()

            soup=bs4.BeautifulSoup(response.content,"html.parser")

            subreddit_data={"name":sr_name,"url":url,"title":soup.title.string if soup.title.string else "No Title"}


            topics=[]

            for heading in soup.find_all(['h1','h2','h3','h4']):
                text=heading.get_text(strip=True)

                if text and len(text)>3:
                    if any(keyword in text.lower() for keyword in ['rise','oil','bullish','stocks','bonds','tax','chips','luxury']):
                        topics.append({"title":text,"subreddit":sr_name,"type":"topic"})
            discussions=[]
            seen_url=set()

            for link in soup.find_all('a',href=True):
                text=link.get_text(strip=True)
                href=link['href']
                if text and len(text)>1 and ('/comments/' in href) and (href not in seen_url):
                    seen_url.add(href)
                    discussions.append({"title":text,"url":href,"type":"discussions"})

            subreddit_data["topic"]=topics
            subreddit_data["discussions"]=discussions


            all_data.append(subreddit_data)
            time.sleep(2)

        except Exception as e:
            print(e)

    return all_data

def save_scraped_data(data, filename_json, filename_csv):
    if not data:
        print(f'No data')
        return


    try:
        with open(filename_json, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=True)
        print(f'Topics saved: {filename_json}')
    except Exception as e:
        print("ERROR:",e)



    try:
        with open(filename_csv, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Subreddit', 'Type', 'Title', 'URL'])

            for subreddit_data in data:
                subreddit = subreddit_data['name']

                for topic in subreddit_data.get('topics', []):
                    writer.writerow([subreddit, topic['type'], topic['title'], ''])

                for discussion in subreddit_data.get('discussions', []):
                    writer.writerow([subreddit, discussion['type'], discussion['title'], discussion['url']])

            print(f'All topics saved to files!')
    except Exception as e:
        print("Error:",e)
a=scrape_reddit(subreddit)
save_scraped_data(a,"smth.json","smth.csv")
a
'''

In [2]:
def scrape_reddit(subreddit: list, total_posts: int):
    all_data = []  #main list
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}  #framework

    for index, url in enumerate(subreddit):
        sr_name = url.split("/")[-2]  #names of subreddit
        after = None
        posts = []
        print(sr_name)

        while len(posts) < total_posts:

            params = {"limit": total_posts}
            #to check the loop may be taking 50 posts then is checking which is the stuff we want etc
            if after:
                params["after"] = after

            try:
                response = requests.get(f"https://www.reddit.com/r/{sr_name}/.json", headers=headers, timeout=15,
                                        params=params)
                res_json = response.json()
                data = res_json["data"]["children"]
                response.raise_for_status()

                for post in data:
                    p = post["data"]

                    comment_url = "https://reddit.com" + p["permalink"] + ".json"

                    comment_response = requests.get(comment_url, headers=headers)
                    comment_json = comment_response.json()
                    comments = []

                    for c in comment_json[1]["data"]["children"]:
                        time.sleep(0.5)
                        if c["kind"] == "t1":   # t1 = comment
                            comments.append(c["data"]["body"])

                    posts.append({"subreddit": sr_name,"title": p["title"],"url": "https://reddit.com" + p["permalink"],"score": p["score"],"num_comments": p["num_comments"],"comments": comments})


                after = res_json["data"]["after"]

                if after is None:
                    break
                time.sleep(1)

                all_data.extend(posts)


            except Exception as e:
                print(e)

    return all_data


def save_scraped_data(data, filename_json, filename_csv):
    if not data:
        print(f'No data')
        return

    try:
        with open(filename_json, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=True)
        print(f'Topics saved: {filename_json}')
    except Exception as e:
        print("ERROR:", e)

    try:
        with open(filename_csv, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Subreddit', 'Type', 'Title', 'URL'])

            for subreddit_data in data:
                subreddit = subreddit_data['name']

                for topic in subreddit_data.get('topics', []):
                    writer.writerow([subreddit, topic['type'], topic['title'], ''])

                for discussion in subreddit_data.get('discussions', []):
                    writer.writerow([subreddit, discussion['type'], discussion['title'], discussion['url']])

            print(f'All topics saved to files!')
    except Exception as e:
        print("Error:", e)

In [5]:
a=scrape_reddit(subreddit,5)

stocks


In [7]:
a

[{'subreddit': 'wallstreetbets',
  'title': 'Weekly Earnings Thread 3/9 - 3/13',
  'url': 'https://reddit.com/r/wallstreetbets/comments/1rmix7j/weekly_earnings_thread_39_313/',
  'score': 10,
  'num_comments': 21,
  'comments': ['Puts after Tuesday? Oracle is going to tank everything like last time ',
   'Anyone have opinions on HPE? Seems like calls as a read-through from Dell’s earnings. That being said I read back through HPE’s last earnings call and they said their gov contracts won’t really hit til later this year.',
   "i still can't believe there's a business of that size in this country that legitimately operates with the name DICKS.",
   "Closed today:\n\n- Calls: IOT • MRVL\n\nOpened today:\n\n- Call: UMAC\n\n*Disclaimer: I'm kinda hand-wavy on this one, subject to change b4 close*.",
   'ORCL puts might print, but premiums are expensive AF. PATH calls? What’s the (regarded) thesis here?',
   'KSS will be a 10 bagger',
   'absolute trash week of earnings.  still, given that, 

In [8]:
a_df=pd.DataFrame(a)
a_df.iloc[1,5]

['Thinking about becoming a gang banger. Any good recommendations on gangs. Looking for something that I know will be around 10 years from now and that has a pastel color because it brings out my eyes ',
 'oil is heading to $100 and this market stays delusional buying every single dip',
 'As a civilization, we should have stuck with robes. \n\nThey look cooler and they’re so much more comfortable.',
 'BERS!!!!!',
 'MMs gonna hate that retail has unlocked the infinite money glitch of selling spx calls and stealing their tendies. ',
 "My Monday spx call better be 10k or I'm coming up there for you god",
 'I can’t believe how many of you actually wanted this because you thought you could “time the chaos” or whatever dumb shit you rationalized in your pea brain',
 'is LITE cooked',
 'Fuck this Im going full bear, 🥭 going full regard',
 'Not seeing a rug pull at vix 26 is insane',
 'USO 0DTE banging ',
 'Why the hell ORCL is up?',
 'Nat gas close to doubled in Europe and UK lol',
 'Leverage

In [9]:
a_df.to_csv(r"C:\Users\Asus\PycharmProjects\test_2\smth.csv")


In [ ]:
#soup.prettify makes html look like normal

#soup.find('<h1,h2,h5,div>') gives that tag line it gives first occurence of h5 soup.find_all('<h5>') gives all occurences
#tags=soup.find_all('<h5>')
#for i in tags:
#   i.text gives text values

#fs=soup.find_all('div',class_='card') {class_ here _ is important as class is a keyword in python}
#for i in fs:
#   i.<'h5','h1'> will give h5 tages or h1 tags
#   i.h5.text gives text
#   i.a

In [15]:
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm

In [26]:
#nltk.download('vader_lexicon')
#nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

True

In [27]:
vad=SentimentIntensityAnalyzer()

In [31]:
s='Hi I am very not happy'
token=nltk.word_tokenize(s)

In [32]:
nltk.pos_tag(token)

[('Hi', 'NNP'),
 ('I', 'PRP'),
 ('am', 'VBP'),
 ('very', 'RB'),
 ('not', 'RB'),
 ('happy', 'JJ')]

In [34]:
path=r"C:\Users\Asus\PycharmProjects\test_2\smth.csv"

In [62]:
df=pd.read_csv(path)

In [70]:
df.iloc[:,6]

0    ['Puts after Tuesday? Oracle is going to tank ...
1    ['Thinking about becoming a gang banger. Any g...
2    ['\n**User Report**| | | |\n:--|:--|:--|:--\n*...
3    ['\n**User Report**| | | |\n:--|:--|:--|:--\n*...
4    ['\n**User Report**| | | |\n:--|:--|:--|:--\n*...
5    ['75% VTI\n\n25% AAPL, AMZN, BLK, BN, GOOGL, I...
6    ['+3% since November which compared to the ind...
7    ['In Trump\'s economy right now, we have jobs ...
8    ['How can this be? I thought the Dow was over ...
9    ['This 90-day free trial of 2026 sucks. How do...
Name: comments, dtype: object

In [69]:
df.iloc[:,6][1]

'[\'Thinking about becoming a gang banger. Any good recommendations on gangs. Looking for something that I know will be around 10 years from now and that has a pastel color because it brings out my eyes \', \'oil is heading to $100 and this market stays delusional buying every single dip\', \'As a civilization, we should have stuck with robes. \\n\\nThey look cooler and they’re so much more comfortable.\', \'BERS!!!!!\', \'MMs gonna hate that retail has unlocked the infinite money glitch of selling spx calls and stealing their tendies. \', "My Monday spx call better be 10k or I\'m coming up there for you god", \'I can’t believe how many of you actually wanted this because you thought you could “time the chaos” or whatever dumb shit you rationalized in your pea brain\', \'is LITE cooked\', \'Fuck this Im going full bear, 🥭 going full regard\', \'Not seeing a rug pull at vix 26 is insane\', \'USO 0DTE banging \', \'Why the hell ORCL is up?\', \'Nat gas close to doubled in Europe and UK l

In [71]:
t=[]
for tokens in df.iloc[:,6]:
        t.extend(nltk.word_tokenize(tokens))

In [72]:
print(len(t),t)

10523 ['[', "'Puts", 'after', 'Tuesday', '?', 'Oracle', 'is', 'going', 'to', 'tank', 'everything', 'like', 'last', 'time', "'", ',', "'Anyone", 'have', 'opinions', 'on', 'HPE', '?', 'Seems', 'like', 'calls', 'as', 'a', 'read-through', 'from', 'Dell', '’', 's', 'earnings', '.', 'That', 'being', 'said', 'I', 'read', 'back', 'through', 'HPE', '’', 's', 'last', 'earnings', 'call', 'and', 'they', 'said', 'their', 'gov', 'contracts', 'won', '’', 't', 'really', 'hit', 'til', 'later', 'this', 'year', '.', "'", ',', '``', 'i', 'still', 'ca', "n't", 'believe', 'there', "'s", 'a', 'business', 'of', 'that', 'size', 'in', 'this', 'country', 'that', 'legitimately', 'operates', 'with', 'the', 'name', 'DICKS', '.', '``', ',', '``', 'Closed', 'today', ':', '\\n\\n-', 'Calls', ':', 'IOT', '•', 'MRVL\\n\\nOpened', 'today', ':', '\\n\\n-', 'Call', ':', 'UMAC\\n\\n', '*', 'Disclaimer', ':', 'I', "'m", 'kinda', 'hand-wavy', 'on', 'this', 'one', ',', 'subject', 'to', 'change', 'b4', 'close', '*', '.', '``', 

In [73]:
tags=nltk.pos_tag(t)
tags

[('[', 'NN'),
 ("'Puts", 'NNS'),
 ('after', 'IN'),
 ('Tuesday', 'NNP'),
 ('?', '.'),
 ('Oracle', 'NNP'),
 ('is', 'VBZ'),
 ('going', 'VBG'),
 ('to', 'TO'),
 ('tank', 'VB'),
 ('everything', 'NN'),
 ('like', 'IN'),
 ('last', 'JJ'),
 ('time', 'NN'),
 ("'", "''"),
 (',', ','),
 ("'Anyone", 'CD'),
 ('have', 'VBP'),
 ('opinions', 'NNS'),
 ('on', 'IN'),
 ('HPE', 'NNP'),
 ('?', '.'),
 ('Seems', 'VBZ'),
 ('like', 'JJ'),
 ('calls', 'NNS'),
 ('as', 'IN'),
 ('a', 'DT'),
 ('read-through', 'NN'),
 ('from', 'IN'),
 ('Dell', 'NNP'),
 ('’', 'NNP'),
 ('s', 'JJ'),
 ('earnings', 'NNS'),
 ('.', '.'),
 ('That', 'IN'),
 ('being', 'VBG'),
 ('said', 'VBD'),
 ('I', 'PRP'),
 ('read', 'VBD'),
 ('back', 'RB'),
 ('through', 'IN'),
 ('HPE', 'NNP'),
 ('’', 'NNP'),
 ('s', 'VBD'),
 ('last', 'JJ'),
 ('earnings', 'NNS'),
 ('call', 'NN'),
 ('and', 'CC'),
 ('they', 'PRP'),
 ('said', 'VBD'),
 ('their', 'PRP$'),
 ('gov', 'NN'),
 ('contracts', 'NNS'),
 ('won', 'VBD'),
 ('’', 'NNP'),
 ('t', 'NNS'),
 ('really', 'RB'),
 ('hit', '

In [74]:
sent_score=[vad.polarity_scores(text) for text in df.iloc[:,6]]

In [75]:
len(sent_score),sent_score
#NOTE : THESE ARE NOT SENTENCE ANALYSIS IT IS WORD ANALYSIS

(10,
 [{'neg': 0.033, 'neu': 0.867, 'pos': 0.099, 'compound': 0.8936},
  {'neg': 0.154, 'neu': 0.766, 'pos': 0.08, 'compound': -0.9995},
  {'neg': 0.147, 'neu': 0.752, 'pos': 0.1, 'compound': -0.9865},
  {'neg': 0.094, 'neu': 0.747, 'pos': 0.159, 'compound': 0.9907},
  {'neg': 0.099, 'neu': 0.814, 'pos': 0.087, 'compound': -0.9708},
  {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0},
  {'neg': 0.113, 'neu': 0.801, 'pos': 0.085, 'compound': -0.9902},
  {'neg': 0.124, 'neu': 0.787, 'pos': 0.089, 'compound': -0.9933},
  {'neg': 0.16, 'neu': 0.745, 'pos': 0.094, 'compound': -0.9901},
  {'neg': 0.165, 'neu': 0.763, 'pos': 0.072, 'compound': -0.9824}])

In [57]:
nltk.pos_tag_sents(t) # cannot use t since the comments are stored in a list so when tokenized it the list square bracket is also tokenized so t becomes a string and non-usable here instead tags can be used but again already converted to tags so no point

TypeError: tokens: expected a list of strings, got a string

In [ ]:
#nltk.word_tokenize(<str>) separates words
#pos=nltk.pos_tag(<str>) categorises into part of speech

#nltk.chunk.ne_chunk(<pos>) groups similar words

#VADER approach bag of words approach, it has values for each word like positive negative or neutral, it does not account for relationship for words it removes stop words